# Supply Chain Demand Forecasting & Inventory Optimization
### Notebook 09 — Streamlit Dashboard Application (Roadmap Items #22-26)

**Author:** Rajveer Singh | M.Sc. Statistics, IIT (BHU)

This notebook builds the project's Streamlit dashboard, step by step, and writes the final
application files directly into `{PROJECT_DIR}/app/`.

**Design decisions, stated upfront:**
1. **The deployed app never loads the 9.4M-row files or the trained model.** Loading
   `feature_engineered_data.csv` or running live inference inside a Streamlit app -- especially on
   free-tier hosting with limited RAM -- would make the app slow or crash it. This notebook does the
   one-time inference work now (same load-saved-model-and-predict pattern as Notebook 08, no
   retraining) and exports a lightweight, app-ready dataset. The deployed app only ever reads small
   precomputed CSVs.
2. **Two new small precomputed artifacts** (an overview-stats summary and a full-history daily
   demand trend, ~1,941 rows) let the Project Overview page render instantly rather than
   aggregating millions of rows on every load.
3. **The app's charts use Plotly, not matplotlib.** Every notebook in this project has used
   matplotlib/seaborn for static analysis -- appropriate there. A deployed interactive dashboard is a
   different context where hover tooltips and zoom matter for a live demo, so `app.py` and every page
   use `plotly.express`/`plotly.graph_objects`.

**What this notebook produces:**
- `app/app.py`, `app/utils.py`, `app/pages/1_Demand_Forecast.py`, `app/pages/2_Inventory_Dashboard.py`,
  `app/pages/3_Model_Insights.py`, `app/requirements.txt` (Roadmap Items #22, #23, #24, #25, #26)
- Three new lightweight data files the app depends on:
  `dashboard_overview_stats.csv`, `daily_demand_trend.csv`, `forecast_vs_actual_final_fold.csv`

Every generated `.py` file was independently syntax-validated with `py_compile` before being
embedded in this notebook -- not just checked as notebook cell content.


## Section 0 — Setup, Paths, Memory Monitoring

In [1]:
import gc
import os
import json
import numpy as np
import pandas as pd
import joblib
import psutil
from sklearn.model_selection import TimeSeriesSplit

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

def show_memory(label=""):
    mem_gb = psutil.Process(os.getpid()).memory_info().rss / 1e9
    print(f"[{label}] Process memory: {mem_gb:.2f} GB")

show_memory("notebook start")

[notebook start] Process memory: 0.25 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Supply_Chain_Demand_Forecasting"
RAW_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
NOTEBOOK_DIR = f"{PROJECT_DIR}/notebooks"
MODELS_DIR = f"{PROJECT_DIR}/models"
APP_DIR = f"{PROJECT_DIR}/app"
PAGES_DIR = f"{APP_DIR}/pages"

os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(PAGES_DIR, exist_ok=True)

CLEANED_DATA_PATH = f"{PROCESSED_DIR}/cleaned_sales_data.csv"
FEATURES_PATH = f"{PROCESSED_DIR}/feature_engineered_data.csv"
BEST_MODEL_PATH = f"{MODELS_DIR}/random_forest_final_fold.pkl"

print("Paths configured. App will be written to:", APP_DIR)

Mounted at /content/drive
Paths configured. App will be written to: /content/drive/MyDrive/Supply_Chain_Demand_Forecasting/app


## Section 1 — Verify Dependencies from Notebooks 01-08

Before building anything, confirm every file this dashboard needs actually exists at the expected
path, and that the saved model loads correctly -- catching a missing or misnamed file here is far
cheaper than debugging it inside a deployed Streamlit app.

In [3]:
required_files = {
    "cleaned_sales_data.csv (Item #6)": CLEANED_DATA_PATH,
    "feature_engineered_data.csv (Item #10)": FEATURES_PATH,
    "baseline_metrics.csv (Item #12)": f"{PROCESSED_DIR}/baseline_metrics.csv",
    "ml_model_metrics.csv (Item #13)": f"{PROCESSED_DIR}/ml_model_metrics.csv",
    "classical_model_metrics.csv (Item #14)": f"{PROCESSED_DIR}/classical_model_metrics.csv",
    "model_comparison_full_catalog.csv (Item #15)": f"{PROCESSED_DIR}/model_comparison_full_catalog.csv",
    "model_comparison_fair_sample.csv (Item #15)": f"{PROCESSED_DIR}/model_comparison_fair_sample.csv",
    "error_analysis_summary.csv (Item #15)": f"{PROCESSED_DIR}/error_analysis_summary.csv",
    "inventory_policy_recommendations.csv (Item #20)": f"{PROCESSED_DIR}/inventory_policy_recommendations.csv",
    "random_forest_final_fold.pkl (Item #17)": BEST_MODEL_PATH,
}

all_present = True
for label, path in required_files.items():
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    if not exists:
        all_present = False
    print(f"[{status}] {label}")

if not all_present:
    raise FileNotFoundError(
        "One or more required files are missing -- re-run the corresponding earlier notebook "
        "before continuing. This dashboard is built entirely on prior notebooks' saved outputs."
    )
print("\nAll required files present.")

[OK] cleaned_sales_data.csv (Item #6)
[OK] feature_engineered_data.csv (Item #10)
[OK] baseline_metrics.csv (Item #12)
[OK] ml_model_metrics.csv (Item #13)
[OK] classical_model_metrics.csv (Item #14)
[OK] model_comparison_full_catalog.csv (Item #15)
[OK] model_comparison_fair_sample.csv (Item #15)
[OK] error_analysis_summary.csv (Item #15)
[OK] inventory_policy_recommendations.csv (Item #20)
[OK] random_forest_final_fold.pkl (Item #17)

All required files present.


In [4]:
# Verify the saved model loads and exposes the expected interface (no retraining, just a load check)
test_model = joblib.load(BEST_MODEL_PATH)
print("Model type:", type(test_model).__name__)
print("Model expects", test_model.n_features_in_, "features")
del test_model
gc.collect()

Model type: RandomForestRegressor
Model expects 27 features


0

**Output interpretation:** `n_features_in_` should equal the length of `FEATURE_COLS` used in
Notebooks 05/07/08 (27 features) -- if it doesn't match, the wrong model file was saved or loaded,
and predictions in Section 2 would silently use misaligned columns.

## Section 2 — Prepare Lightweight App-Ready Datasets

### 2.1 — Overview Statistics (single-row summary)

In [5]:
overview_cols = ['store_id', 'item_id', 'cat_id', 'date']
overview_chunks = []
for chunk in pd.read_csv(CLEANED_DATA_PATH, usecols=overview_cols, parse_dates=['date'],
                          dtype={'store_id': 'category', 'item_id': 'category', 'cat_id': 'category'},
                          chunksize=2_000_000):
    overview_chunks.append(chunk[['store_id', 'item_id', 'cat_id']].drop_duplicates())

overview_ids = pd.concat(overview_chunks, ignore_index=True).drop_duplicates()
del overview_chunks
gc.collect()

# Date range read separately and cheaply (usecols=['date'] only, no need to hold IDs alongside it)
date_range_df = pd.read_csv(CLEANED_DATA_PATH, usecols=['date'], parse_dates=['date'])
n_days = date_range_df['date'].nunique()
del date_range_df
gc.collect()

total_rows = int(pd.read_csv(FEATURES_PATH, usecols=['date'], parse_dates=['date']).shape[0])

In [6]:
comparison_a = pd.read_csv(f"{PROCESSED_DIR}/model_comparison_full_catalog.csv")
best_row = comparison_a.sort_values('avg_wmape_pct').iloc[0]

inventory_df = pd.read_csv(f"{PROCESSED_DIR}/inventory_policy_recommendations.csv",
                            usecols=['risk_tier'], dtype={'risk_tier': 'category'})
n_high_risk = int((inventory_df['risk_tier'] == 'high_stockout_risk').sum())
del inventory_df
gc.collect()

overview_stats = pd.DataFrame([{
    'total_rows': total_rows,
    'n_days': n_days,
    'n_stores': overview_ids['store_id'].nunique(),
    'n_items': overview_ids['item_id'].nunique(),
    'n_categories': overview_ids['cat_id'].nunique(),
    'best_model': best_row['model'],
    'best_model_improvement_pct': round(float(best_row['pct_improvement_vs_baseline']), 2),
    'n_high_stockout_risk': n_high_risk,
}])

OVERVIEW_OUT = f"{PROCESSED_DIR}/dashboard_overview_stats.csv"
overview_stats.to_csv(OVERVIEW_OUT, index=False)

del overview_ids
gc.collect()
print("Saved dashboard_overview_stats.csv")
overview_stats

Saved dashboard_overview_stats.csv


,total_rows,n_days,n_stores,n_items,n_categories,best_model,best_model_improvement_pct,n_high_stockout_risk
0,9415478,1941,2,3049,3,random_forest,17.65,1525


### 2.2 — Full-History Daily Demand Trend (~1,941 rows)

In [7]:
daily_trend_chunks = []
for chunk in pd.read_csv(CLEANED_DATA_PATH, usecols=['date', 'units_sold'], parse_dates=['date'],
                          dtype={'units_sold': 'float32'}, chunksize=2_000_000):
    daily_trend_chunks.append(chunk.groupby('date')['units_sold'].sum())

daily_trend = pd.concat(daily_trend_chunks).groupby(level=0).sum().reset_index()
daily_trend.columns = ['date', 'units_sold']
daily_trend = daily_trend.sort_values('date')

del daily_trend_chunks
gc.collect()

DAILY_TREND_OUT = f"{PROCESSED_DIR}/daily_demand_trend.csv"
daily_trend.to_csv(DAILY_TREND_OUT, index=False)

print("Saved daily_demand_trend.csv —", daily_trend.shape)
daily_trend.head()

Saved daily_demand_trend.csv — (1941, 2)


,date,units_sold
0,2011-01-29,6893.0
1,2011-01-30,6842.0
2,2011-01-31,4638.0
3,2011-02-01,5309.0
4,2011-02-02,4324.0


### 2.3 — Forecast vs. Actual, Full Catalog, Final Fold (~170K rows)

Reusing the exact fold-boundary and chunk-filtered-load pattern established in Notebooks 07 and 08,
and the same saved Random Forest model -- no retraining, one inference pass, immediately trimmed
down to only the columns the dashboard actually needs.

In [8]:
FORECAST_HORIZON = 28
unique_dates = np.sort(pd.read_csv(CLEANED_DATA_PATH, usecols=['date'], parse_dates=['date'])['date'].unique())
tscv = TimeSeriesSplit(n_splits=5, test_size=FORECAST_HORIZON)
final_fold = list(tscv.split(unique_dates))[-1]
test_start_date = unique_dates[final_fold[1][0]]
test_end_date = unique_dates[final_fold[1][-1]]

del unique_dates
gc.collect()

print(f"Final fold test window: {pd.Timestamp(test_start_date).date()} -> {pd.Timestamp(test_end_date).date()}")

Final fold test window: 2016-04-25 -> 2016-05-22


In [9]:
FEATURE_COLS = [
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    'rolling_std_7', 'rolling_std_14', 'rolling_std_28',
    'rolling_median_7', 'rolling_median_14', 'rolling_median_28',
    'ema_7', 'ema_14',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_weekend', 'snap', 'has_event', 'snap_lead',
    'sell_price', 'price_rolling_mean_28', 'price_relative', 'price_changed',
]
TARGET = 'units_sold'
EXTRA_COLS = ['store_id', 'item_id', 'cat_id', 'dept_id']

USE_COLS = list(dict.fromkeys(['date'] + EXTRA_COLS + [TARGET] + FEATURE_COLS))

int8_cols = ['is_weekend', 'snap', 'has_event', 'price_changed']
float32_cols = [c for c in FEATURE_COLS if c not in int8_cols] + [TARGET]
dtype_map = {c: 'int8' for c in int8_cols}
dtype_map.update({c: 'float32' for c in float32_cols})
dtype_map.update({'store_id': 'category', 'item_id': 'category', 'cat_id': 'category', 'dept_id': 'category'})

filtered_chunks = []
for chunk in pd.read_csv(FEATURES_PATH, usecols=USE_COLS, dtype=dtype_map, parse_dates=['date'],
                          chunksize=2_000_000):
    mask = (chunk['date'] >= test_start_date) & (chunk['date'] <= test_end_date)
    filtered_chunks.append(chunk.loc[mask])

final_fold_df = pd.concat(filtered_chunks, ignore_index=True)
del filtered_chunks
gc.collect()

show_memory("after final-fold filtered load")
print("Final-fold slice:", final_fold_df.shape)

[after final-fold filtered load] Process memory: 0.95 GB
Final-fold slice: (170744, 33)


In [10]:
best_model = joblib.load(BEST_MODEL_PATH)
final_fold_df['forecast'] = np.clip(best_model.predict(final_fold_df[FEATURE_COLS]), 0, None).astype('float32')
final_fold_df = final_fold_df.rename(columns={TARGET: 'actual'})

del best_model
gc.collect()

# Trim to only what the dashboard needs -- dropping all 27 feature columns now that predictions exist
dashboard_export_cols = ['date', 'store_id', 'item_id', 'cat_id', 'dept_id', 'actual', 'forecast']
forecast_vs_actual = final_fold_df[dashboard_export_cols].copy()

FORECAST_OUT = f"{PROCESSED_DIR}/forecast_vs_actual_final_fold.csv"
forecast_vs_actual.to_csv(FORECAST_OUT, index=False)

print("Saved forecast_vs_actual_final_fold.csv —", forecast_vs_actual.shape)
forecast_vs_actual.head()

Saved forecast_vs_actual_final_fold.csv — (170744, 7)


,date,store_id,item_id,cat_id,dept_id,actual,forecast
0,2016-04-25,CA_1,FOODS_1_001,FOODS,FOODS_1,2.0,0.975751
1,2016-04-26,CA_1,FOODS_1_001,FOODS,FOODS_1,0.0,0.982442
2,2016-04-27,CA_1,FOODS_1_001,FOODS,FOODS_1,0.0,0.967749
3,2016-04-28,CA_1,FOODS_1_001,FOODS,FOODS_1,0.0,0.939689
4,2016-04-29,CA_1,FOODS_1_001,FOODS,FOODS_1,0.0,0.853327


In [11]:
del final_fold_df, forecast_vs_actual
gc.collect()
show_memory("after Section 2 cleanup")

[after Section 2 cleanup] Process memory: 0.96 GB


## Section 3 — Write Application Files

Every file written below was independently syntax-validated with `py_compile` before being placed
in this notebook. `%%writefile` cells write their content verbatim -- changing directory into
`APP_DIR` first means the relative paths below (`app.py`, `pages/...`) land in the correct project
folder regardless of where this notebook happens to be running from.

In [12]:
os.chdir(APP_DIR)
print("Working directory set to:", os.getcwd())

Working directory set to: /content/drive/MyDrive/Supply_Chain_Demand_Forecasting/app


### 3.1 — `utils.py` (Roadmap Item #26)

Shared caching and data-loading functions every page imports from -- keeps the column contracts
and path configuration in one place.

In [13]:
%%writefile utils.py
"""
Shared data-loading and caching utilities for the Supply Chain Demand Forecasting dashboard.

Every page imports from this module rather than reading CSVs directly -- this keeps the
caching strategy, path configuration, and column contracts in one place (Roadmap Item #26).
No model is loaded or trained here: every function reads a precomputed CSV produced by the
project's notebooks (Items #6, #10, #12-15, #20), consistent with this project's
notebook-to-notebook independence pattern established from Notebook 04 onward.
"""

import os
import pandas as pd
import streamlit as st

# ============================================================
# PROJECT PATHS -- same convention used in every notebook
# ============================================================
PROJECT_DIR = "/content/drive/MyDrive/Supply_Chain_Demand_Forecasting"
RAW_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
MODELS_DIR = f"{PROJECT_DIR}/models"

# When deployed outside of Colab (e.g. Streamlit Community Cloud, or run locally), the app
# falls back to a relative "data/processed" folder next to app.py -- this makes the same
# codebase runnable both from Google Drive (Colab) and from a cloned GitHub repo.
if not os.path.isdir(PROCESSED_DIR):
    PROCESSED_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "data", "processed")


def _read_csv(filename, **kwargs):
    """Small wrapper so every loader shares one consistent path-join and error message."""
    path = os.path.join(PROCESSED_DIR, filename)
    if not os.path.exists(path):
        st.error(f"Missing data file: {filename}. Expected at: {path}")
        st.stop()
    return pd.read_csv(path, **kwargs)


# ============================================================
# OVERVIEW PAGE DATA
# ============================================================
@st.cache_data
def load_overview_stats():
    """Single-row summary stats (Roadmap Item #22 support) -- tiny file, loads instantly."""
    return _read_csv("dashboard_overview_stats.csv").iloc[0].to_dict()


@st.cache_data
def load_daily_trend():
    """Full-history aggregated daily demand (~1,941 rows) -- cheap to load and plot."""
    df = _read_csv("daily_demand_trend.csv", parse_dates=["date"])
    return df


# ============================================================
# DEMAND FORECAST PAGE DATA
# ============================================================
@st.cache_data
def load_forecast_vs_actual():
    """Daily actual vs. forecast for the full catalog, final fold only (~170K rows).
    Precomputed once by the build notebook using the saved Random Forest model --
    the app never runs inference live."""
    df = _read_csv(
        "forecast_vs_actual_final_fold.csv",
        parse_dates=["date"],
        dtype={"store_id": "category", "item_id": "category", "cat_id": "category", "dept_id": "category"},
    )
    return df


# ============================================================
# INVENTORY OPTIMIZATION PAGE DATA
# ============================================================
@st.cache_data
def load_inventory_recommendations():
    """Per-store-item safety stock, reorder point, EOQ, and risk tier (Roadmap Item #20)."""
    df = _read_csv(
        "inventory_policy_recommendations.csv",
        dtype={"store_id": "category", "item_id": "category", "cat_id": "category", "dept_id": "category",
               "risk_tier": "category"},
    )
    return df


# ============================================================
# MODEL INSIGHTS PAGE DATA
# ============================================================
@st.cache_data
def load_comparison_a():
    """Full-catalog, 5-fold Baseline vs. ML comparison (Roadmap Item #15, Comparison A)."""
    return _read_csv("model_comparison_full_catalog.csv")


@st.cache_data
def load_comparison_b():
    """Fair three-way comparison: Baseline vs. ML vs. Classical, same 180 series, final fold
    (Roadmap Item #15, Comparison B)."""
    return _read_csv("model_comparison_fair_sample.csv")


@st.cache_data
def load_ml_fold_metrics():
    """Fold-by-fold ML metrics, used for the stability chart (Roadmap Item #13)."""
    return _read_csv("ml_model_metrics.csv")


@st.cache_data
def load_error_analysis():
    """Error breakdown by category/store/month/event (Roadmap Item #15, Enhancement #3)."""
    return _read_csv("error_analysis_summary.csv")


# ============================================================
# SHARED FILTER HELPERS
# ============================================================
def get_category_options(df, col="cat_id"):
    return sorted(df[col].dropna().unique().tolist())


def get_store_options(df, col="store_id"):
    return sorted(df[col].dropna().unique().tolist())


def to_csv_download(df):
    """Convert a dataframe to UTF-8 CSV bytes for st.download_button, used consistently
    across every page's downloadable-output requirement."""
    return df.to_csv(index=False).encode("utf-8")


Writing utils.py


### 3.2 — `app.py` (Roadmap Item #22)

The dashboard's landing page -- project overview, KPI cards, and the full-history demand trend.

In [14]:
%%writefile app.py
"""
Supply Chain Demand Forecasting & Inventory Optimization -- Dashboard Entry Point (Roadmap Item #22)

Landing page: project overview, headline KPI cards, and the full-history demand trend.
Detailed forecasting, inventory, and model-comparison views live in the pages/ folder
(Roadmap Items #23-25), following Streamlit's standard multipage app convention.
"""

import streamlit as st
import plotly.express as px

from utils import load_overview_stats, load_daily_trend

st.set_page_config(
    page_title="Supply Chain Demand Forecasting",
    page_icon="📦",
    layout="wide",
)

st.title("📦 Supply Chain Demand Forecasting & Inventory Optimization")
st.caption("M5 (Walmart) dataset, subsetted to CA_1 and TX_1 -- end-to-end forecasting and inventory policy")

st.markdown("""
This dashboard presents an end-to-end demand forecasting and inventory optimization pipeline,
built on real Walmart sales data (the M5 forecasting competition dataset). Every number shown
here comes from a saved, reproducible notebook pipeline -- no data is generated live by this app.

**Use the sidebar to navigate:**
- **Demand Forecast** -- actual vs. forecast by product, with category/store filters and search
- **Inventory Dashboard** -- safety stock, reorder points, and stockout/overstock risk
- **Model Insights** -- baseline vs. machine learning vs. classical model comparison
""")

stats = load_overview_stats()

st.markdown("### Project Snapshot")
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Records Analyzed", f"{int(stats['total_rows']):,}")
col2.metric("Date Range (Days)", f"{int(stats['n_days']):,}")
col3.metric("Stores Covered", f"{int(stats['n_stores'])}")
col4.metric("Product Categories", f"{int(stats['n_categories'])}")

col5, col6, col7, col8 = st.columns(4)
col5.metric("Unique Products", f"{int(stats['n_items']):,}")
col6.metric("Best Model (Full Catalog)", stats['best_model'])
col7.metric("Improvement Over Baseline", f"{stats['best_model_improvement_pct']:.1f}%")
col8.metric("High-Risk Products (Stockout)", f"{int(stats['n_high_stockout_risk']):,}")

st.markdown("### Total Daily Demand — Full History")
daily_trend = load_daily_trend()

fig = px.line(
    daily_trend, x="date", y="units_sold",
    labels={"date": "Date", "units_sold": "Total Units Sold"},
    title="Total Daily Units Sold — CA_1 + TX_1 Combined (Full History)",
)
fig.update_layout(height=420, hovermode="x unified")
st.plotly_chart(fig, use_container_width=True)

st.markdown("""
---
**Methodology summary:** demand was forecast using a Random Forest model, selected after
comparing 5 machine learning models, 2 baseline methods, and 2 classical time series models
(SARIMA, Prophet) across walk-forward cross-validation. Safety stock and reorder points were
then derived analytically from the winning model's forecast error distribution. See the
**Model Insights** page for the full comparison, and the **Inventory Dashboard** page for the
resulting reorder recommendations.
""")


Writing app.py


### 3.3 — `pages/1_Demand_Forecast.py` (Roadmap Item #23)

Category/store filters, product search, actual-vs-forecast chart, category-level aggregate view,
and a downloadable CSV of the filtered selection.

In [15]:
%%writefile pages/1_Demand_Forecast.py
"""
Demand Forecast page (Roadmap Item #23) -- interactive actual vs. forecast viewer.

Loads the precomputed forecast_vs_actual_final_fold.csv (Random Forest predictions on the
final walk-forward fold, full catalog) -- no model inference happens in this app.
"""

import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

from utils import load_forecast_vs_actual, get_category_options, get_store_options, to_csv_download

st.set_page_config(page_title="Demand Forecast", page_icon="📈", layout="wide")
st.title("📈 Demand Forecast — Actual vs. Predicted")
st.caption("Random Forest predictions on the final walk-forward fold (28 days), full catalog")

df = load_forecast_vs_actual()

# ------------------------------------------------------------
# Sidebar filters (Roadmap requirement: category/store filters, product search)
# ------------------------------------------------------------
st.sidebar.header("Filters")

categories = get_category_options(df, "cat_id")
selected_categories = st.sidebar.multiselect("Category", categories, default=categories)

stores = get_store_options(df, "store_id")
selected_stores = st.sidebar.multiselect("Store", stores, default=stores)

filtered_df = df[df["cat_id"].isin(selected_categories) & df["store_id"].isin(selected_stores)]

st.sidebar.markdown("---")
st.sidebar.subheader("Product Search")
item_options = sorted(filtered_df["item_id"].unique().tolist())
if len(item_options) == 0:
    st.warning("No products match the current filters. Adjust the category/store selection.")
    st.stop()

selected_item = st.sidebar.selectbox("Search / Select Product (item_id)", item_options)
store_options_for_item = sorted(filtered_df.loc[filtered_df["item_id"] == selected_item, "store_id"].unique().tolist())
selected_store_for_item = st.sidebar.selectbox("Store for Selected Product", store_options_for_item)

# ------------------------------------------------------------
# KPI cards for the filtered selection
# ------------------------------------------------------------
st.markdown("### Filtered Selection Summary")
kpi1, kpi2, kpi3 = st.columns(3)
kpi1.metric("Products in Selection", f"{filtered_df['item_id'].nunique():,}")
kpi2.metric("Total Actual Units (Final Fold)", f"{filtered_df['actual'].sum():,.0f}")
kpi3.metric("Total Forecast Units (Final Fold)", f"{filtered_df['forecast'].sum():,.0f}")

# ------------------------------------------------------------
# Product-level actual vs. forecast chart
# ------------------------------------------------------------
st.markdown(f"### Actual vs. Forecast — Product `{selected_item}` at `{selected_store_for_item}`")

product_df = filtered_df[
    (filtered_df["item_id"] == selected_item) & (filtered_df["store_id"] == selected_store_for_item)
].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(x=product_df["date"], y=product_df["actual"], mode="lines+markers", name="Actual"))
fig.add_trace(go.Scatter(x=product_df["date"], y=product_df["forecast"], mode="lines+markers", name="Forecast"))
fig.update_layout(
    title=f"Daily Units Sold — {selected_item} @ {selected_store_for_item}",
    xaxis_title="Date", yaxis_title="Units Sold", height=420, hovermode="x unified",
)
st.plotly_chart(fig, use_container_width=True)

# ------------------------------------------------------------
# Category-level aggregate view
# ------------------------------------------------------------
st.markdown("### Category-Level Actual vs. Forecast (Filtered Selection)")
category_agg = filtered_df.groupby(["date", "cat_id"], observed=True)[["actual", "forecast"]].sum().reset_index()

fig2 = px.line(
    category_agg, x="date", y="actual", color="cat_id",
    labels={"date": "Date", "actual": "Total Actual Units", "cat_id": "Category"},
    title="Total Actual Demand by Category (Final Fold)",
)
fig2.update_layout(height=400, hovermode="x unified")
st.plotly_chart(fig2, use_container_width=True)

# ------------------------------------------------------------
# Downloadable output
# ------------------------------------------------------------
st.markdown("### Download Filtered Data")
st.download_button(
    label="Download Filtered Forecast Data (CSV)",
    data=to_csv_download(filtered_df),
    file_name="filtered_forecast_vs_actual.csv",
    mime="text/csv",
)


Writing pages/1_Demand_Forecast.py


### 3.4 — `pages/2_Inventory_Dashboard.py` (Roadmap Item #24)

Safety stock, reorder points, risk-tier distribution, and a reorder alert list -- all read from
Notebook 08's saved recommendations, nothing recomputed here.

In [16]:
%%writefile pages/2_Inventory_Dashboard.py
"""
Inventory Dashboard page (Roadmap Item #24) -- safety stock, reorder points, and risk tiers.

Loads inventory_policy_recommendations.csv (Roadmap Item #20), computed analytically in
Notebook 08 from the Random Forest forecast's error distribution -- this app does not
recompute any inventory formulas, it only visualizes the saved recommendations.
"""

import streamlit as st
import plotly.express as px

from utils import load_inventory_recommendations, get_category_options, get_store_options, to_csv_download

st.set_page_config(page_title="Inventory Dashboard", page_icon="📦", layout="wide")
st.title("📦 Inventory Optimization Dashboard")
st.caption("Safety stock, reorder points, and EOQ -- derived analytically from forecast uncertainty (Item #19)")

st.info(
    "This dataset does not include actual stock-on-hand levels. Safety stock, reorder point, "
    "and risk tier below are derived analytically from the forecast and its error distribution, "
    "using explicit business assumptions (7-day lead time, 95% service level) -- not from a live "
    "inventory feed. See Notebook 08 for the full methodology.",
    icon="ℹ️",
)

df = load_inventory_recommendations()

# ------------------------------------------------------------
# Sidebar filters
# ------------------------------------------------------------
st.sidebar.header("Filters")
categories = get_category_options(df, "cat_id")
selected_categories = st.sidebar.multiselect("Category", categories, default=categories)

stores = get_store_options(df, "store_id")
selected_stores = st.sidebar.multiselect("Store", stores, default=stores)

risk_tiers = sorted(df["risk_tier"].dropna().unique().tolist())
selected_risk_tiers = st.sidebar.multiselect("Risk Tier", risk_tiers, default=risk_tiers)

filtered_df = df[
    df["cat_id"].isin(selected_categories)
    & df["store_id"].isin(selected_stores)
    & df["risk_tier"].isin(selected_risk_tiers)
]

if len(filtered_df) == 0:
    st.warning("No products match the current filters. Adjust the selection in the sidebar.")
    st.stop()

# ------------------------------------------------------------
# KPI cards
# ------------------------------------------------------------
st.markdown("### Inventory KPIs (Filtered Selection)")
kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1.metric("Products in Selection", f"{len(filtered_df):,}")
kpi2.metric("Total Safety Stock (units)", f"{filtered_df['safety_stock'].sum():,.0f}")
kpi3.metric("Avg. Reorder Point", f"{filtered_df['reorder_point'].mean():,.1f}")
kpi4.metric("High Stockout-Risk Products", f"{(filtered_df['risk_tier'] == 'high_stockout_risk').sum():,}")

# ------------------------------------------------------------
# Safety stock by category
# ------------------------------------------------------------
st.markdown("### Total Safety Stock by Category")
safety_by_cat = filtered_df.groupby("cat_id", observed=True)["safety_stock"].sum().reset_index()
fig1 = px.bar(
    safety_by_cat.sort_values("safety_stock", ascending=True), x="safety_stock", y="cat_id",
    orientation="h", labels={"safety_stock": "Total Safety Stock (units)", "cat_id": "Category"},
    title="Total Safety Stock Required by Category",
)
fig1.update_layout(height=380)
st.plotly_chart(fig1, use_container_width=True)

# ------------------------------------------------------------
# Risk tier distribution
# ------------------------------------------------------------
col_left, col_right = st.columns(2)

with col_left:
    st.markdown("### Risk Tier Distribution")
    risk_counts = filtered_df["risk_tier"].value_counts().reset_index()
    risk_counts.columns = ["risk_tier", "count"]
    color_map = {
        "high_stockout_risk": "#C44E52", "moderate_risk": "#DD8452",
        "overstock_risk": "#4C72B0", "no_demand": "#8C8C8C",
    }
    fig2 = px.pie(
        risk_counts, names="risk_tier", values="count", color="risk_tier",
        color_discrete_map=color_map, title="Product-Store Count by Risk Tier",
    )
    fig2.update_layout(height=380)
    st.plotly_chart(fig2, use_container_width=True)

with col_right:
    st.markdown("### Reorder Point Distribution")
    fig3 = px.histogram(
        filtered_df, x="reorder_point", nbins=40,
        labels={"reorder_point": "Reorder Point (units)"},
        title="Distribution of Reorder Points",
    )
    fig3.update_layout(height=380)
    st.plotly_chart(fig3, use_container_width=True)

# ------------------------------------------------------------
# Reorder alert table
# ------------------------------------------------------------
st.markdown("### Reorder Alert List — Highest-Priority Products")
top_n = st.slider("Number of products to show", min_value=5, max_value=50, value=20, step=5)
alerts = filtered_df.sort_values("reorder_point", ascending=False).head(top_n)[
    ["store_id", "item_id", "cat_id", "avg_daily_demand_forecast", "safety_stock", "reorder_point",
     "eoq", "risk_tier"]
]
st.dataframe(alerts, use_container_width=True)

# ------------------------------------------------------------
# Downloadable output
# ------------------------------------------------------------
st.markdown("### Download Filtered Inventory Recommendations")
st.download_button(
    label="Download Filtered Inventory Data (CSV)",
    data=to_csv_download(filtered_df),
    file_name="filtered_inventory_recommendations.csv",
    mime="text/csv",
)


Writing pages/2_Inventory_Dashboard.py


### 3.5 — `pages/3_Model_Insights.py` (Roadmap Item #25)

Comparison A, Comparison B, fold stability, and the detailed error analysis breakdown -- all read
from Notebook 07's saved outputs.

In [17]:
%%writefile pages/3_Model_Insights.py
"""
Model Insights page (Roadmap Item #25) -- baseline vs. ML vs. classical model comparison.

Loads the saved comparison tables from Notebook 07 (Roadmap Item #15) directly -- no metric
is recomputed here, only visualized.
"""

import streamlit as st
import plotly.express as px

from utils import load_comparison_a, load_comparison_b, load_ml_fold_metrics, load_error_analysis, to_csv_download

st.set_page_config(page_title="Model Insights", page_icon="🔬", layout="wide")
st.title("🔬 Model Comparison & Insights")
st.caption("Baseline vs. Machine Learning vs. Classical Time Series models (Roadmap Item #15)")

comparison_a = load_comparison_a()
comparison_b = load_comparison_b()

# ------------------------------------------------------------
# Comparison A: full catalog, 5-fold
# ------------------------------------------------------------
st.markdown("### Comparison A — Full Catalog, 5-Fold Walk-Forward Average")
st.caption("Baseline vs. ML models, evaluated across all ~6,000 series and all 5 folds")

fig1 = px.bar(
    comparison_a.sort_values("avg_wmape_pct"), x="avg_wmape_pct", y="model", orientation="h",
    color="beats_baseline",
    color_discrete_map={True: "#4C72B0", False: "#C44E52"},
    labels={"avg_wmape_pct": "Average WMAPE (%)", "model": "Model", "beats_baseline": "Beats Baseline"},
    title="Average WMAPE by Model — Full Catalog, 5-Fold",
)
fig1.update_layout(height=420)
st.plotly_chart(fig1, use_container_width=True)

st.dataframe(comparison_a.sort_values("avg_wmape_pct"), use_container_width=True)

best_row = comparison_a.sort_values("avg_wmape_pct").iloc[0]
st.success(
    f"**Best full-catalog model: {best_row['model']}** — {best_row['avg_wmape_pct']:.2f}% WMAPE, "
    f"a {best_row['pct_improvement_vs_baseline']:.1f}% relative improvement over the best baseline."
)

st.markdown("---")

# ------------------------------------------------------------
# Comparison B: fair three-way
# ------------------------------------------------------------
st.markdown("### Comparison B — Fair Three-Way Comparison")
st.caption("Baseline vs. ML vs. Classical, same 180-series sample, same final fold (apples-to-apples)")

family_filter = st.multiselect(
    "Filter by model family", options=sorted(comparison_b["family"].unique().tolist()),
    default=sorted(comparison_b["family"].unique().tolist()),
)
filtered_b = comparison_b[comparison_b["family"].isin(family_filter)]

fig2 = px.bar(
    filtered_b.sort_values("pooled_wmape_pct"), x="pooled_wmape_pct", y="model", orientation="h",
    color="family",
    color_discrete_map={"Baseline": "#C44E52", "ML": "#4C72B0", "Classical": "#55A868"},
    labels={"pooled_wmape_pct": "Pooled WMAPE (%)", "model": "Model", "family": "Model Family"},
    title="Pooled WMAPE — Baseline vs. ML vs. Classical (Same Sample, Final Fold)",
)
fig2.update_layout(height=420)
st.plotly_chart(fig2, use_container_width=True)

st.dataframe(filtered_b.sort_values("pooled_wmape_pct"), use_container_width=True)

st.markdown("---")

# ------------------------------------------------------------
# Fold stability
# ------------------------------------------------------------
st.markdown("### Fold-by-Fold Stability (ML Models)")
st.caption("Lower variance across folds indicates a more production-reliable model")

ml_fold_metrics = load_ml_fold_metrics()
pivot = ml_fold_metrics.pivot(index="fold", columns="model", values="wmape") * 100
pivot_long = pivot.reset_index().melt(id_vars="fold", var_name="model", value_name="wmape_pct")

fig3 = px.line(
    pivot_long, x="fold", y="wmape_pct", color="model", markers=True,
    labels={"fold": "Fold", "wmape_pct": "WMAPE (%)", "model": "Model"},
    title="WMAPE by Fold — ML Model Stability",
)
fig3.update_layout(height=400)
st.plotly_chart(fig3, use_container_width=True)

st.markdown("---")

# ------------------------------------------------------------
# Error analysis breakdown
# ------------------------------------------------------------
st.markdown("### Detailed Error Analysis")
st.caption("Where the best full-catalog model struggles most (Roadmap Item #15, Enhancement #3)")

error_df = load_error_analysis()
breakdown_choice = st.selectbox(
    "Breakdown dimension", sorted(error_df["breakdown"].unique().tolist())
)
breakdown_df = error_df[error_df["breakdown"] == breakdown_choice].sort_values("wmape_pct", ascending=False)

fig4 = px.bar(
    breakdown_df, x="wmape_pct", y="group_value", orientation="h",
    labels={"wmape_pct": "WMAPE (%)", "group_value": breakdown_choice.title()},
    title=f"WMAPE by {breakdown_choice.title()}",
)
fig4.update_layout(height=380)
st.plotly_chart(fig4, use_container_width=True)

st.dataframe(breakdown_df, use_container_width=True)

# ------------------------------------------------------------
# Downloadable outputs
# ------------------------------------------------------------
st.markdown("### Download Comparison Data")
dl_col1, dl_col2, dl_col3 = st.columns(3)
dl_col1.download_button(
    "Comparison A (CSV)", data=to_csv_download(comparison_a),
    file_name="model_comparison_full_catalog.csv", mime="text/csv",
)
dl_col2.download_button(
    "Comparison B (CSV)", data=to_csv_download(comparison_b),
    file_name="model_comparison_fair_sample.csv", mime="text/csv",
)
dl_col3.download_button(
    "Error Analysis (CSV)", data=to_csv_download(error_df),
    file_name="error_analysis_summary.csv", mime="text/csv",
)


Writing pages/3_Model_Insights.py


### 3.6 — `requirements.txt`

Deliberately minimal -- no scikit-learn, since the deployed app never loads the model directly
(Section 2 already did the one-time inference and exported the results).

In [18]:
%%writefile requirements.txt
streamlit>=1.32.0
pandas>=2.0.0
plotly>=5.18.0


Writing requirements.txt


## Section 4 — Verify Written Application Files

Confirm every file landed in the right place with the expected size, and restore the working
directory to avoid side effects on any later notebook.

In [19]:
expected_files = [
    "app.py", "utils.py", "requirements.txt",
    "pages/1_Demand_Forecast.py", "pages/2_Inventory_Dashboard.py", "pages/3_Model_Insights.py",
]

print("Files written to", APP_DIR, ":\n")
for f in expected_files:
    full_path = os.path.join(APP_DIR, f)
    if os.path.exists(full_path):
        size_kb = os.path.getsize(full_path) / 1024
        print(f"  [OK] {f} ({size_kb:.1f} KB)")
    else:
        print(f"  [MISSING] {f}")

os.chdir(PROJECT_DIR)
print("\nWorking directory restored to:", os.getcwd())

Files written to /content/drive/MyDrive/Supply_Chain_Demand_Forecasting/app :

  [OK] app.py (2.9 KB)
  [OK] utils.py (4.9 KB)
  [OK] requirements.txt (0.0 KB)
  [OK] pages/1_Demand_Forecast.py (4.2 KB)
  [OK] pages/2_Inventory_Dashboard.py (5.3 KB)
  [OK] pages/3_Model_Insights.py (5.3 KB)

Working directory restored to: /content/drive/MyDrive/Supply_Chain_Demand_Forecasting


## Section 5 — How to Run This App

### Run locally (after cloning the GitHub repo)
```bash
cd app
pip install -r requirements.txt
streamlit run app.py
```

### Run from Google Drive (Colab-mounted path)
```bash
cd /content/drive/MyDrive/Supply_Chain_Demand_Forecasting/app
streamlit run app.py
```

### Deploy to Streamlit Community Cloud
1. Push the `app/` folder (and `data/processed/`, or a subset of the small CSVs it depends on) to
   the GitHub repository.
2. On [share.streamlit.io](https://share.streamlit.io), point the deployment at `app/app.py`.
3. Streamlit Cloud will install from `app/requirements.txt` automatically -- no scikit-learn or
   large model file needed, since all inference was precomputed in Section 2 of this notebook.

**Note:** the `utils.py` path fallback (see Section 3.1) means the same codebase works whether
`PROJECT_DIR` resolves to a mounted Google Drive path or the app is deployed from a cloned repo with
data sitting in a relative `data/processed/` folder next to `app.py`.


---
## Roadmap Status
- **Item #22** (`app/app.py`) — complete: project overview, KPI cards, full-history demand trend
- **Item #23** (`app/pages/1_Demand_Forecast.py`) — complete: category/store filters, product
  search, actual-vs-forecast chart, downloadable CSV
- **Item #24** (`app/pages/2_Inventory_Dashboard.py`) — complete: safety stock, reorder points,
  risk-tier distribution, reorder alert list, downloadable CSV
- **Item #25** (`app/pages/3_Model_Insights.py`) — complete: Comparison A/B, fold stability, error
  analysis breakdown, downloadable CSVs
- **Item #26** (`app/utils.py`) — complete: shared caching/loading functions, Colab/local/deployed
  path fallback
- Three new lightweight data files: `dashboard_overview_stats.csv`, `daily_demand_trend.csv`,
  `forecast_vs_actual_final_fold.csv` — all precomputed here, no live inference in the deployed app

## Key Takeaways
- The dashboard reuses every prior notebook's saved output without retraining or recomputing any
  model or metric -- consistent with this project's notebook-independence pattern established since
  Notebook 04.
- Every `.py` file was syntax-validated independently with `py_compile` before being embedded in this
  notebook, in addition to the notebook-level validation applied to every prior notebook in this
  project.
- The app's data footprint is intentionally minimal (three small CSVs plus five already-saved ones)
  rather than shipping multi-million-row files or a live model into a deployed environment.

## What's Next — Item #28 Preview
`README.md`: the professional GitHub README tying together every notebook, the dashboard, and the
full project narrative — beginning Phase 6.
